# Alarm clock

**Time:** 5 minutes

**What you build:** an analog alarm clock app. The notebook creates a RocketRide app, draws the clock (a ticking face, a time picker and an alarm that flashes and beeps), then deploys it to RocketRide Cloud and publishes it to your own desktop.

**Done when:** the last cell prints **Open your clock** and the link shows your clock ticking on cloud.rocketride.ai.

> All you need is Python 3.10+, your RocketRide API key, and a developer id for your organization (Stage 1 tells you how to claim one). Click **Run All**, or run the cells from top to bottom with **Shift+Enter**. Every cell is safe to run again.

## Setup

Installs the RocketRide SDK into this notebook's Python the first time you run it. After that it takes a second.

In [ ]:
# Setup: installs the RocketRide SDK if it's missing or too old. Safe to re-run.
import importlib, re, site, subprocess, sys
from importlib import metadata

if sys.version_info < (3, 10):
    raise RuntimeError(f"This notebook needs Python 3.10 or newer, and this kernel is {sys.version.split()[0]}. "
                       "Pick a newer Python with Select Kernel in the top right.")

SDK, MIN_VERSION = "rocketride>=1.4.0", (1, 4, 0)

def sdk_version():
    try:
        return tuple(int(n) for n in re.findall(r"\d+", metadata.version("rocketride"))[:3])
    except metadata.PackageNotFoundError:
        return None

def pip_install(*extra):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", SDK, *extra],
                          capture_output=True, text=True)

if (sdk_version() or (0,)) < MIN_VERSION:
    print("Installing the RocketRide SDK. This takes about a minute, once.")
    result = pip_install()
    if result.returncode and "externally-managed" in result.stderr:
        # Homebrew and Linux system Pythons block pip; install for this user instead.
        result = pip_install("--user", "--break-system-packages")
        site.addsitedir(site.getusersitepackages())
    if result.returncode:
        raise RuntimeError(f"Couldn't install the RocketRide SDK:\n{result.stderr}")
    importlib.invalidate_caches()
    if "rocketride" in sys.modules:
        raise RuntimeError("Updated the RocketRide SDK. Restart the kernel, then run the notebook again.")
print("\033[32m✅ ready\033[0m")

## Config

Reads your RocketRide settings from the environment or a `.env` file next to this notebook. If your API key isn't set, it asks for it. Nothing is saved.

In [ ]:
# Config: the ONLY cell that reads environment variables. Every later cell uses CONFIG.
import getpass, os
from pathlib import Path

def read_dotenv(path=Path(".env")):
    values = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            key, sep, value = line.strip().partition("=")
            if sep and key.strip().startswith("ROCKETRIDE_"):
                values[key.strip()] = value.strip().strip("\"'")
    return values

env = read_dotenv()  # a .env file next to this notebook...
env.update({k: v for k, v in os.environ.items() if k.startswith("ROCKETRIDE_")})  # ...env wins
env.setdefault("ROCKETRIDE_URI", "https://api.rocketride.ai")
if not env.get("ROCKETRIDE_APIKEY"):
    previous = globals().get("CONFIG", {}).get("apikey")  # don't re-prompt on re-run
    env["ROCKETRIDE_APIKEY"] = previous or getpass.getpass("RocketRide API key: ").strip()

CONFIG = {
    "uri": env["ROCKETRIDE_URI"],
    "apikey": env["ROCKETRIDE_APIKEY"],
    "env": env,
}
print("Server: ", CONFIG["uri"])
print("API key:", "set" if CONFIG["apikey"] else "MISSING")

In [ ]:
# Connect: one shared RocketRide client for every cell below. Safe to re-run.
from rocketride import RocketRideClient

if "client" not in globals() or not client.is_connected():
    client = RocketRideClient(uri=CONFIG["uri"], auth=CONFIG["apikey"], env=CONFIG["env"])
    await client.connect()
print("Connected to", CONFIG["uri"])

async def start(pipe_path):
    """Start a pipeline, or attach to it if it's already running. Returns the task token."""
    result = await client.use(filepath=str(pipe_path), use_existing=True)
    return result["token"]

## Stage 1: Create the app (1 min)

**Goal:** create the app from RocketRide's app template. Every app's id is `<developer id>.<name>`, so this cell first reads your organization's developer id, then writes the app's files under `build/apps/alarm_clock/`. Nothing is installed on your machine: RocketRide Cloud builds the app in Stage 3.

**Watch for:** `App id:` followed by your app id, and the list of files the template wrote.

In [ ]:
# Stage 1: create the app from RocketRide's app template. Safe to re-run.
import json, shutil
from pathlib import Path

WORKSPACE = Path("build").resolve()
APP_ROOT = WORKSPACE / "apps" / "alarm_clock"

def developer_id():
    org = (client.get_account_info() or {}).get("organization") or {}
    return org.get("developerId")

if not developer_id():
    # Claimed in the extension while this notebook was connected? Reconnect to pick it up.
    await client.disconnect()
    await client.connect()
if not developer_id():
    raise RuntimeError("Your organization has no developer id yet. In VS Code, open the RocketRide "
                       "App Builder, go to the Deploy tab and claim one (letters and underscores, "
                       "for example acme_labs). Then run this cell again.")
APP_ID = f"{developer_id()}.alarm_clock"

# An app made under a different developer id can't deploy here, so start that one over.
manifest = APP_ROOT / "package.json"
if manifest.exists() and json.loads(manifest.read_text(encoding="utf-8"))["appManifest"]["id"] != APP_ID:
    shutil.rmtree(APP_ROOT)
if not APP_ROOT.exists():
    WORKSPACE.mkdir(exist_ok=True)
    await client.deploy.create_app("alarm_clock", workspace_root=str(WORKSPACE), developer_id=developer_id(),
                                   display_name="Alarm Clock", install=False)

print("App id:", APP_ID)
for path in sorted(APP_ROOT.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(APP_ROOT).as_posix())

In [ ]:
# Skip ahead: run this only if the stage above didn't work
import json, shutil
from pathlib import Path

WORKSPACE = Path("build").resolve()
APP_ROOT = WORKSPACE / "apps" / "alarm_clock"
await client.disconnect()
await client.connect()  # fresh identity, in case the developer id was just claimed
dev_id = ((client.get_account_info() or {}).get("organization") or {}).get("developerId")
if not dev_id:
    raise RuntimeError("Still no developer id. Claim one in the App Builder's Deploy tab first.")
APP_ID = f"{dev_id}.alarm_clock"

shutil.rmtree(APP_ROOT, ignore_errors=True)  # start from a clean template
WORKSPACE.mkdir(exist_ok=True)
await client.deploy.create_app("alarm_clock", workspace_root=str(WORKSPACE), developer_id=dev_id,
                               display_name="Alarm Clock", install=False)
print("Created a fresh app:", APP_ID, "Carry on with Stage 2.")

## Stage 2: Draw the clock (1 min)

**Goal:** replace the template's hello screen with the clock. The app is a small React component in `src/App.tsx`: an SVG face with 60 ticks, 12 numerals and three hands that move every second, a digital readout, and an alarm. Set a time and press **Set alarm**: when the clock gets there, the face flashes and it beeps until you press **Stop**. The cell also writes the README that appears on the app's store listing.

**Watch for:** `Wrote src/App.tsx` and `Wrote README.md`.

In [ ]:
# Stage 2: write the clock into src/App.tsx. Safe to re-run.
from pathlib import Path

APP_ROOT = Path("build/apps/alarm_clock").resolve()

APP_TSX = r"""/**
 * Alarm Clock: an analog clock with one alarm, rendered by the RocketRide shell.
 */

import React, { useEffect, useRef, useState } from 'react';
import { AppLayout } from 'shell';

const SIZE = 260;
const MID = SIZE / 2;

type HandProps = { turns: number; length: number; width: number; color: string };

/** One hand. `turns` is how far round the face it points: 0 is 12 o'clock, 0.5 is 6. */
const Hand: React.FC<HandProps> = ({ turns, length, width, color }) => {
	const angle = turns * 2 * Math.PI;
	const x = MID + length * Math.sin(angle);
	const y = MID - length * Math.cos(angle);
	return <line x1={MID} y1={MID} x2={x} y2={y} stroke={color} strokeWidth={width} strokeLinecap="round" />;
};

/** The face: 60 ticks, 12 numerals, and the three hands. */
const Face: React.FC<{ now: Date; flash: boolean }> = ({ now, flash }) => {
	const seconds = now.getSeconds();
	const minutes = now.getMinutes() + seconds / 60;
	const hours = (now.getHours() % 12) + minutes / 60;

	const ticks = [];
	for (let i = 0; i < 60; i++) {
		const angle = (i / 60) * 2 * Math.PI;
		const inner = i % 5 === 0 ? MID - 18 : MID - 10;
		ticks.push(
			<line key={i} stroke="currentColor" strokeWidth={i % 5 === 0 ? 3 : 1}
				x1={MID + inner * Math.sin(angle)} y1={MID - inner * Math.cos(angle)}
				x2={MID + (MID - 4) * Math.sin(angle)} y2={MID - (MID - 4) * Math.cos(angle)} />,
		);
	}

	const numerals = [];
	for (let n = 1; n <= 12; n++) {
		const angle = (n / 12) * 2 * Math.PI;
		numerals.push(
			<text key={n} x={MID + (MID - 36) * Math.sin(angle)} y={MID - (MID - 36) * Math.cos(angle)}
				textAnchor="middle" dominantBaseline="central" fontSize={18} fill="currentColor">{n}</text>,
		);
	}

	let fill = 'var(--rr-bg-paper, #ffffff)';
	if (flash) {
		fill = 'var(--rr-color-error, #e5484d)';
	}

	return (
		<svg width={SIZE} height={SIZE} viewBox={`0 0 ${SIZE} ${SIZE}`} style={{ color: 'var(--rr-text-primary, #111)' }}>
			<circle cx={MID} cy={MID} r={MID - 2} fill={fill} stroke="currentColor" strokeWidth={3} />
			{ticks}
			{numerals}
			<Hand turns={hours / 12} length={MID * 0.5} width={6} color="currentColor" />
			<Hand turns={minutes / 60} length={MID * 0.75} width={4} color="currentColor" />
			<Hand turns={seconds / 60} length={MID * 0.85} width={2} color="var(--rr-color-error, #e5484d)" />
			<circle cx={MID} cy={MID} r={5} fill="currentColor" />
		</svg>
	);
};

/** Plays one short beep. */
function beep(audio: AudioContext): void {
	const tone = audio.createOscillator();
	tone.frequency.value = 880;
	tone.connect(audio.destination);
	tone.start();
	tone.stop(audio.currentTime + 0.25);
}

const Clock: React.FC = () => {
	const [now, setNow] = useState(new Date());
	const [alarm, setAlarm] = useState('07:00');
	const [isArmed, setIsArmed] = useState(false);
	const [isRinging, setIsRinging] = useState(false);
	const audio = useRef<AudioContext | null>(null);

	// Tick once a second.
	useEffect(() => {
		const timer = setInterval(() => {
			setNow(new Date());
		}, 1000);
		return () => {
			clearInterval(timer);
		};
	}, []);

	// Ring when the clock reaches the alarm time. It fires once, then disarms.
	const hhmm = now.toTimeString().slice(0, 5);
	useEffect(() => {
		if (isArmed && hhmm === alarm) {
			setIsArmed(false);
			setIsRinging(true);
		}
	}, [isArmed, hhmm, alarm]);

	// Beep every second while ringing.
	useEffect(() => {
		if (!isRinging || !audio.current) {
			return;
		}
		const sound = audio.current;
		beep(sound);
		const timer = setInterval(() => {
			beep(sound);
		}, 1000);
		return () => {
			clearInterval(timer);
		};
	}, [isRinging]);

	// Browsers only allow sound after a click, so the audio starts when you set the alarm.
	const toggleAlarm = () => {
		if (!audio.current) {
			audio.current = new AudioContext();
		}
		void audio.current.resume();
		setIsArmed(!isArmed);
	};

	let status = 'Alarm off';
	if (isArmed) {
		status = `Alarm set for ${alarm}`;
	}
	if (isRinging) {
		status = 'Wake up!';
	}

	return (
		<div style={styles.wrap}>
			<Face now={now} flash={isRinging && now.getSeconds() % 2 === 0} />
			<div style={styles.digital}>{now.toLocaleTimeString()}</div>
			<div style={styles.row}>
				<input type="time" value={alarm} onChange={(e) => { setAlarm(e.target.value); }} style={styles.input} />
				<button onClick={toggleAlarm} style={styles.button}>{isArmed ? 'Turn off' : 'Set alarm'}</button>
				{isRinging && <button onClick={() => { setIsRinging(false); }} style={styles.button}>Stop</button>}
			</div>
			<div style={styles.status}>{status}</div>
		</div>
	);
};

const styles: Record<string, React.CSSProperties> = {
	wrap: { display: 'flex', flexDirection: 'column', alignItems: 'center', gap: 16, padding: 32, fontFamily: 'var(--rr-font-family, system-ui)' },
	digital: { fontSize: 28, fontVariantNumeric: 'tabular-nums', color: 'var(--rr-text-primary, #111)' },
	row: { display: 'flex', gap: 8, alignItems: 'center' },
	input: { fontSize: 16, padding: '6px 8px' },
	button: { fontSize: 14, padding: '6px 14px', cursor: 'pointer' },
	status: { fontSize: 13, color: 'var(--rr-text-secondary, #666)' },
};

/** Root view: the shell frame with the clock inside. */
const App: React.FC = () => {
	return (
		<AppLayout showStatus>
			<Clock />
		</AppLayout>
	);
};

export default App;
"""

README_MD = """# Alarm Clock

An analog alarm clock. Pick a time, press **Set alarm**, and when the clock gets there the face flashes and it beeps until you press **Stop**.

Built in the RocketRide alarm-clock workshop.
"""

(APP_ROOT / "src" / "App.tsx").write_text(APP_TSX, encoding="utf-8", newline="\n")
(APP_ROOT / "README.md").write_text(README_MD, encoding="utf-8", newline="\n")
print("Wrote src/App.tsx")
print("Wrote README.md")

In [ ]:
# Skip ahead: run this only if the stage above didn't work
from pathlib import Path

APP_ROOT = Path("build/apps/alarm_clock").resolve()

APP_TSX = r"""/**
 * Alarm Clock: an analog clock with one alarm, rendered by the RocketRide shell.
 */

import React, { useEffect, useRef, useState } from 'react';
import { AppLayout } from 'shell';

const SIZE = 260;
const MID = SIZE / 2;

type HandProps = { turns: number; length: number; width: number; color: string };

/** One hand. `turns` is how far round the face it points: 0 is 12 o'clock, 0.5 is 6. */
const Hand: React.FC<HandProps> = ({ turns, length, width, color }) => {
	const angle = turns * 2 * Math.PI;
	const x = MID + length * Math.sin(angle);
	const y = MID - length * Math.cos(angle);
	return <line x1={MID} y1={MID} x2={x} y2={y} stroke={color} strokeWidth={width} strokeLinecap="round" />;
};

/** The face: 60 ticks, 12 numerals, and the three hands. */
const Face: React.FC<{ now: Date; flash: boolean }> = ({ now, flash }) => {
	const seconds = now.getSeconds();
	const minutes = now.getMinutes() + seconds / 60;
	const hours = (now.getHours() % 12) + minutes / 60;

	const ticks = [];
	for (let i = 0; i < 60; i++) {
		const angle = (i / 60) * 2 * Math.PI;
		const inner = i % 5 === 0 ? MID - 18 : MID - 10;
		ticks.push(
			<line key={i} stroke="currentColor" strokeWidth={i % 5 === 0 ? 3 : 1}
				x1={MID + inner * Math.sin(angle)} y1={MID - inner * Math.cos(angle)}
				x2={MID + (MID - 4) * Math.sin(angle)} y2={MID - (MID - 4) * Math.cos(angle)} />,
		);
	}

	const numerals = [];
	for (let n = 1; n <= 12; n++) {
		const angle = (n / 12) * 2 * Math.PI;
		numerals.push(
			<text key={n} x={MID + (MID - 36) * Math.sin(angle)} y={MID - (MID - 36) * Math.cos(angle)}
				textAnchor="middle" dominantBaseline="central" fontSize={18} fill="currentColor">{n}</text>,
		);
	}

	let fill = 'var(--rr-bg-paper, #ffffff)';
	if (flash) {
		fill = 'var(--rr-color-error, #e5484d)';
	}

	return (
		<svg width={SIZE} height={SIZE} viewBox={`0 0 ${SIZE} ${SIZE}`} style={{ color: 'var(--rr-text-primary, #111)' }}>
			<circle cx={MID} cy={MID} r={MID - 2} fill={fill} stroke="currentColor" strokeWidth={3} />
			{ticks}
			{numerals}
			<Hand turns={hours / 12} length={MID * 0.5} width={6} color="currentColor" />
			<Hand turns={minutes / 60} length={MID * 0.75} width={4} color="currentColor" />
			<Hand turns={seconds / 60} length={MID * 0.85} width={2} color="var(--rr-color-error, #e5484d)" />
			<circle cx={MID} cy={MID} r={5} fill="currentColor" />
		</svg>
	);
};

/** Plays one short beep. */
function beep(audio: AudioContext): void {
	const tone = audio.createOscillator();
	tone.frequency.value = 880;
	tone.connect(audio.destination);
	tone.start();
	tone.stop(audio.currentTime + 0.25);
}

const Clock: React.FC = () => {
	const [now, setNow] = useState(new Date());
	const [alarm, setAlarm] = useState('07:00');
	const [isArmed, setIsArmed] = useState(false);
	const [isRinging, setIsRinging] = useState(false);
	const audio = useRef<AudioContext | null>(null);

	// Tick once a second.
	useEffect(() => {
		const timer = setInterval(() => {
			setNow(new Date());
		}, 1000);
		return () => {
			clearInterval(timer);
		};
	}, []);

	// Ring when the clock reaches the alarm time. It fires once, then disarms.
	const hhmm = now.toTimeString().slice(0, 5);
	useEffect(() => {
		if (isArmed && hhmm === alarm) {
			setIsArmed(false);
			setIsRinging(true);
		}
	}, [isArmed, hhmm, alarm]);

	// Beep every second while ringing.
	useEffect(() => {
		if (!isRinging || !audio.current) {
			return;
		}
		const sound = audio.current;
		beep(sound);
		const timer = setInterval(() => {
			beep(sound);
		}, 1000);
		return () => {
			clearInterval(timer);
		};
	}, [isRinging]);

	// Browsers only allow sound after a click, so the audio starts when you set the alarm.
	const toggleAlarm = () => {
		if (!audio.current) {
			audio.current = new AudioContext();
		}
		void audio.current.resume();
		setIsArmed(!isArmed);
	};

	let status = 'Alarm off';
	if (isArmed) {
		status = `Alarm set for ${alarm}`;
	}
	if (isRinging) {
		status = 'Wake up!';
	}

	return (
		<div style={styles.wrap}>
			<Face now={now} flash={isRinging && now.getSeconds() % 2 === 0} />
			<div style={styles.digital}>{now.toLocaleTimeString()}</div>
			<div style={styles.row}>
				<input type="time" value={alarm} onChange={(e) => { setAlarm(e.target.value); }} style={styles.input} />
				<button onClick={toggleAlarm} style={styles.button}>{isArmed ? 'Turn off' : 'Set alarm'}</button>
				{isRinging && <button onClick={() => { setIsRinging(false); }} style={styles.button}>Stop</button>}
			</div>
			<div style={styles.status}>{status}</div>
		</div>
	);
};

const styles: Record<string, React.CSSProperties> = {
	wrap: { display: 'flex', flexDirection: 'column', alignItems: 'center', gap: 16, padding: 32, fontFamily: 'var(--rr-font-family, system-ui)' },
	digital: { fontSize: 28, fontVariantNumeric: 'tabular-nums', color: 'var(--rr-text-primary, #111)' },
	row: { display: 'flex', gap: 8, alignItems: 'center' },
	input: { fontSize: 16, padding: '6px 8px' },
	button: { fontSize: 14, padding: '6px 14px', cursor: 'pointer' },
	status: { fontSize: 13, color: 'var(--rr-text-secondary, #666)' },
};

/** Root view: the shell frame with the clock inside. */
const App: React.FC = () => {
	return (
		<AppLayout showStatus>
			<Clock />
		</AppLayout>
	);
};

export default App;
"""

README_MD = """# Alarm Clock

An analog alarm clock. Pick a time, press **Set alarm**, and when the clock gets there the face flashes and it beeps until you press **Stop**.

Built in the RocketRide alarm-clock workshop.
"""

(APP_ROOT / "src" / "App.tsx").write_text(APP_TSX, encoding="utf-8", newline="\n")
(APP_ROOT / "README.md").write_text(README_MD, encoding="utf-8", newline="\n")
print("Wrote the finished clock. Carry on with Stage 3.")

## Stage 3: Deploy and open it (3 min)

**Goal:** check the app, upload it, and publish it. The app's source goes up to RocketRide Cloud, which installs its packages and builds it. That takes a minute or two, and the cell prints each build step as it happens. When the build is done, the cell publishes the app to **@me**, your own desktop, and prints a link that opens it.

**Watch for:** `Published v1 to @me` and `Open your clock:` with the link. Open it, set an alarm one minute ahead, and wait for it to ring. Run the cell again after you change the clock: each run publishes the next version.

In [ ]:
# Stage 3: upload the app, wait for the cloud build, publish it to your desktop. Safe to re-run.
import asyncio

UI_URL = "https://cloud.rocketride.ai"  # where RocketRide apps open in the browser

report = await client.deploy.verify_app(str(APP_ROOT), workspace_root=str(WORKSPACE))
for check in report.checks:
    if not check.ok:
        print("❌", check.id, check.note)
if not report.ok:
    raise RuntimeError("The app didn't pass its checks. Fix the items above, then run this cell again.")
print(f"Checked: {report.file_count} files")

added = await client.deploy.add_app(str(APP_ROOT), workspace_root=str(WORKSPACE), comment="Alarm clock workshop")
version = added["artifact"]["registryVersion"]
print(f"Uploaded v{version}. Building in RocketRide Cloud...")

status, last = "", ""
for _ in range(60):  # up to 5 minutes
    rail = await client.list_deployments(APP_ID)
    entry = next((v for v in rail if v.get("registryVersion") == version), {})
    status = entry.get("buildStatus", "")
    step = entry.get("buildPhase") or status or "queued"
    if step != last:
        print("  ", step)
        last = step
    if status in ("ok", "failed"):
        break
    await asyncio.sleep(5)

if status == "failed":
    log = (await client.build_log(APP_ID, version)).get("log") or ""
    print(log[-3000:])
    raise RuntimeError(f"The cloud build of v{version} failed. The end of its build log is above.")
if status != "ok":
    raise RuntimeError(f"v{version} is still building. Wait a minute, then run the skip-ahead cell below.")

await client.publish_app(APP_ID, version, "@me")
print(f"Published v{version} to @me")
print()
print("Open your clock:", f"{UI_URL}/?appid={APP_ID}")
print("It's also on your RocketRide desktop the next time it refreshes.")

In [ ]:
# Skip ahead: run this only if the stage above didn't work
UI_URL = "https://cloud.rocketride.ai"

rail = await client.list_deployments(APP_ID)
built = [v for v in rail if v.get("buildStatus") == "ok"]  # newest first
if not built:
    states = ", ".join(f'v{v.get("registryVersion")}: {v.get("buildPhase") or v.get("buildStatus")}' for v in rail)
    raise RuntimeError(f"No version has finished building yet ({states or 'nothing uploaded'}). "
                       "Wait a minute and run this cell again, or run Stage 3 again.")
version = built[0]["registryVersion"]
await client.publish_app(APP_ID, version, "@me")
print(f"Published v{version} to @me")
print("Open your clock:", f"{UI_URL}/?appid={APP_ID}")

## What's next

- **Docs:** [docs.rocketride.org](https://docs.rocketride.org)
- **Help and community:** [RocketRide Discord](https://discord.gg/PMXrtenMsY)
- **Next module:** [Integrate RocketRide into an existing app](https://github.com/rocketride-org/rocketride-workshops/blob/main/modules/notepad-integration/notepad-integration.ipynb)